# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [16]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    'table': ['employees', 'salary', 'studies'],
    'definition': [
        'Employee information, name, address, title',
        'Salary details for each year',
        'Educational studies, name of the institution, years completed'
    ]
}

df = pd.DataFrame(data)
print(df)



       table                                         definition
0  employees         Employee information, name, address, title
1     salary                       Salary details for each year
2    studies  Educational studies, name of the institution, ...


In [17]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [14]:
print(text_tables)

employees: Employee information, name...
salary: Salary details for each year
studies: Educational studies, name of the institution, type of studies, level


In [18]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [20]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Who is the most senior employee?")

In [24]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees", "salary"]
}
```


In [21]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question= "What are the top 3 largest offices?")

In [22]:
print(return_OAI(pqt3))

```json
{
    "tables": ["employees"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [ ]:
# Definition of the tables.
import pandas as pd

# --- Example 1: Bank Transactions ---
bank_data = {
    'table': ['customers', 'accounts', 'transactions'],
    'definition': [
        'name, address, ID, contact info',
        'account number, type, balance',
        'amount, date, type, account ID'
    ]
}

bank_df = pd.DataFrame(bank_data)
print("Bank Example:")
print(bank_df)
print("\n")


# --- Example 2: Retail Store ---
store_data = {
    'table': ['products', 'orders', 'inventory', 'customers'],
    'definition': [
        'product, category, price, and SKU',
        'order ID, product ID, quantity, date',
        'Stock levels, product',
        'Name, loyalty ID, contact details'
    ]
}

store_df = pd.DataFrame(store_data)
print("Store Example:")
print(store_df)
print("\n")

# --- Example 3: Shakira Fan Club ---
fanclub_data = {
    'table': ['members', 'events', 'merchandise', 'donations'],
    'definition': [
        'Name, country, membership level',
        'Concerts, meetups, online sessions, location, date',
        'Merchandise item name, price,stock',
        'Donor ID, amount,campaign supported'
    ]
}

fanclub_df = pd.DataFrame(fanclub_data)
print("Shakira Fan Club Example:")
print(fanclub_df)

Bank Example:
          table                       definition
0     customers  name, address, ID, contact info
1      accounts    account number, type, balance
2  transactions   amount, date, type, account ID


Store Example:
       table                            definition
0   products     product, category, price, and SKU
1     orders  order ID, product ID, quantity, date
2  inventory                 Stock levels, product
3  customers     Name, loyalty ID, contact details


Shakira Fan Club Example:
         table                                         definition
0      members                    Name, country, membership level
1       events  Concerts, meetups, online sessions, location, ...
2  merchandise                 Merchandise item name, price,stock
3    donations                Donor ID, amount,campaign supported


In [30]:
text_tablesbank_df = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in bank_df.iterrows()])
print("Bank example:\n", text_tablesbank_df)

text_tablesstore_df = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in store_df.iterrows()])
print("Store example:\n", text_tablesstore_df)

text_tablesfanclub_df = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in fanclub_df.iterrows()])
print("Store example:\n", text_tablesfanclub_df)

Bank example:
 customers: name, address, ID, contact info
accounts: account number, type, balance
transactions: amount, date, type, account ID
Store example:
 products: product, category, price, and SKU
orders: order ID, product ID, quantity, date
inventory: Stock levels, product
customers: Name, loyalty ID, contact details
Store example:
 members: Name, country, membership level
events: Concerts, meetups, online sessions, location, date
merchandise: Merchandise item name, price,stock
donations: Donor ID, amount,campaign supported


In [32]:
pqe1 = prompt_question_tables.format(tables=text_tablesbank_df, question="Which transaction could be a fraud and who onws them?")
print(return_OAI(pqe1))

{
    "tables": ["customers", "accounts", "transactions"]
}


In [33]:
pqe2 = prompt_question_tables.format(tables=text_tablesstore_df, question="What are the top 3 sellers and in which areas?")
print(return_OAI(pqe2))

```json
{
    "tables": ["products", "orders"]
}
```


In [34]:
pqe3 = prompt_question_tables.format(tables=text_tablesfanclub_df, question="Which fans have attended all of the events?")
print(return_OAI(pqe3))

```json
{
    "tables": ["members", "events"]
}
```


Observations:
-The scenarios worked reliably, producing clear SQL queries aligned with the table structure. 
-When asked in the bank to provide the tables to detect fraud it came with a good assuption, although there was no field related to this directly. 
-I learned that GPT performs best when table names and relationships are concrete and descriptive; ambiguity in schema design increases the risk of invented or mismatched SQL elements.